### Author: Celia Baumhoer

In [ ]:
!pip install folium

In [52]:
import re
from pathlib import Path

import folium
import matplotlib.colors as mcolors
import rasterio
from folium.features import GeoJson, GeoJsonTooltip
from pyproj import Transformer
from shapely.geometry import box
from shapely.ops import transform

In [60]:
in_path = Path(
    "/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/Landsat/fsc_raw_switzerland"
)  # your folder
pattern = "*.tif"

# Regex to extract the year from filenames like:
# LC08_L2SP_195028_20210118_...
year_regex = re.compile(r".*_(\d{4})\d{4}_")

In [61]:
records = []

for tif in in_path.rglob("*.tif"):
    m = year_regex.match(tif.name)
    if not m:
        continue
    year = int(m.group(1))

    with rasterio.open(tif) as src:
        left, bottom, right, top = src.bounds
        geom = box(left, bottom, right, top)

        # Transform extent to EPSG:4326 for GeoJSON
        if src.crs.to_epsg() != 4326:
            transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
            geom = transform(transformer.transform, geom)

    records.append({"year": year, "geometry": geom})

In [62]:
geojson_features = []
for rec in records:
    coords = [
        [
            [rec["geometry"].bounds[0], rec["geometry"].bounds[1]],
            [rec["geometry"].bounds[0], rec["geometry"].bounds[3]],
            [rec["geometry"].bounds[2], rec["geometry"].bounds[3]],
            [rec["geometry"].bounds[2], rec["geometry"].bounds[1]],
            [rec["geometry"].bounds[0], rec["geometry"].bounds[1]],
        ]
    ]
    geojson_features.append(
        {
            "type": "Feature",
            "properties": {"year": rec["year"]},
            "geometry": {"type": "Polygon", "coordinates": coords},
        }
    )

geojson = {"type": "FeatureCollection", "features": geojson_features}

In [63]:
# Generate colors for each year
years = sorted({f["properties"]["year"] for f in geojson_features})
n_years = len(years)

# Choose a colormap (Set1, tab10, etc.)
cmap = plt.get_cmap("Set1", n_years)

# Map year to hex color
year_to_color = {year: mcolors.to_hex(cmap(i)) for i, year in enumerate(years)}


def style_function(feature):
    year = feature["properties"]["year"]
    return {"fillColor": year_to_color[year], "color": "black", "weight": 1, "fillOpacity": 0.4}

In [64]:
# ---------------------------
# Create Folium map
# ---------------------------
def style_function(feature):
    year = feature["properties"]["year"]
    return {
        "color": year_to_color[year],  # edge color
        "weight": 2,  # line thickness
        "fill": False,  # do not fill
        "fillOpacity": 0,  # ensures no fill
    }


# Center map roughly on your extents
if geojson_features:
    b = geojson_features[0]["geometry"]["coordinates"][0]
    center_lat = sum(p[1] for p in b) / len(b)
    center_lon = sum(p[0] for p in b) / len(b)
else:
    center_lat, center_lon = 46, 11  # fallback

m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles="CartoDB positron")


# Add extents
geojson_layer = GeoJson(
    geojson,
    style_function=style_function,
    tooltip=GeoJsonTooltip(fields=["year"], aliases=["Year"]),
)
folium.GeoJson(
    geojson,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["year"], aliases=["Year"]),
).add_to(m)
m
# Save map as HTML
# m.save("/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/figures/training_data_extent.html")